# AI工学101 — 第38回

## 分類モデルを本気で比較する：Logistic Regression → Random Forest → Gradient Boosting

よし、レベル。今日は**分類モデル比較の実戦回**だぞ。🔥

前回は回帰で、

```text
Baseline
↓
複数モデル
↓
Cross Validation
↓
公平な比較
```

をやった。

今日は分類版。

ただし、分類は回帰よりさらに厄介だ。

なぜなら、

> **「どの失敗を重く見るか」によって、良いモデルが変わるから。**

例えば病気の検出。

```text
実際に病気
↓
病気ではないと予測
```

これは危険。

一方、

```text
病気ではない
↓
病気かもしれないと予測
```

なら、追加検査というコストで済むかもしれない。

両方とも「間違い」ではある。

でも、**同じ重さではない。**

今日は、

> **Accuracyが高いモデル＝良いモデル、ではない**

を実験で理解するぞ。ふふふ。

---

# 🎯 今日のゴール

今日は次ができるようになる。

* Baseline分類器を作る
* Logistic Regressionを比較対象として使う
* Random Forestを使う
* Gradient Boostingを使う
* Accuracy / Precision / Recall / F1を区別する
* ROC-AUCを理解する
* PR-AUCを理解する
* クラス不均衡の問題を理解する
* `predict_proba()` を使える
* 分類閾値を調整できる
* Cross Validationで公平にモデル比較できる

---

# 📖 講義：約20〜25分

## 1. 分類問題とは？

例えば、

```text
メール
↓
spam / not spam
```

。

あるいは、

```text
ユーザー情報
↓
購入する / しない
```

。

今日は、

```text
0
1
```

の二値分類を中心に考える。

---

# 2. 混同行列

分類の結果は、4種類に分けられる。

```text
                 実際
             0          1

予測 0      TN         FN

予測 1      FP         TP
```

。

それぞれ。

### TP

True Positive。

```text
予測 = 1
実際 = 1
```

。

正しくPositive。

---

### TN

True Negative。

```text
予測 = 0
実際 = 0
```

。

正しくNegative。

---

### FP

False Positive。

```text
予測 = 1
実際 = 0
```

。

偽陽性。

---

### FN

False Negative。

```text
予測 = 0
実際 = 1
```

。

偽陰性。

---

# 🧠 3. Accuracy

一番わかりやすい。

```text
正しく分類できた数
÷
全データ
```

。

```python
from sklearn.metrics import accuracy_score
```

。

例えば100件中90件正解。

```text
Accuracy = 90%
```

。

簡単。

でも問題がある。

---

# 🚨 クラス不均衡

例えば、

```text
99人
↓
健康

1人
↓
病気
```

。

ここでモデルが、

```text
全員健康
```

と予測したら。

```text
Accuracy = 99%
```

。

でも、

> **病気の人を1人も発見していない。**

Accuracyだけ見ると、

```text
99%！
```

。

実態は、

```text
役に立たない
```

。

これがクラス不均衡。

---

# 💻 実習1：データを用意する

今日は `breast_cancer` データセットを使う。

```python
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(
    as_frame=True
)

X = data.data
y = data.target

print(
    X.shape
)

print(
    y.value_counts()
)
```

---

# 💻 実習2：Train / Test Split

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
```

ここで、

```python
stratify=y
```

。

これは、

> **Train / Testでクラス比率をなるべく保つ**

ため。

分類問題では重要。

---

# 🧠 4. Baseline

まず、

```python
DummyClassifier
```

。

```python
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(
    strategy="most_frequent"
)
```

。

これは、

```text
一番多いクラス
```

を常に予測する。

```python
baseline.fit(
    X_train,
    y_train
)

pred = baseline.predict(
    X_test
)
```

評価。

```python
from sklearn.metrics import accuracy_score

print(
    accuracy_score(
        y_test,
        pred
    )
)
```

。

ここで、

> Baselineが思ったより高いAccuracy

を出すことがある。

それがクラス不均衡の罠。

---

# 🧠 5. Precision

Precisionは、

> **「Positiveだと言ったもののうち、本当にPositiveだった割合」**

。

```text
TP
──────
TP + FP
```

。

例えば、

```text
病気だ！
```

と100人に言った。

実際に病気だったのは80人。

```text
Precision = 80%
```

。

FPを減らしたいとき重要。

---

# 🧠 6. Recall

Recallは、

> **実際のPositiveをどれだけ見つけられたか**

。

```text
TP
──────
TP + FN
```

。

例えば病気の人が100人いる。

そのうち95人発見した。

```text
Recall = 95%
```

。

FNを減らしたいとき重要。

---

# 🧠 PrecisionとRecallのトレードオフ

例えば、

```text
ちょっとでも怪しい
↓
Positive
```

にすると、

```text
Recall ↑
```

しやすい。

でも、

```text
Precision ↓
```

しやすい。

逆に、

```text
かなり確実
↓
Positive
```

にすると、

```text
Precision ↑
```

しやすい。

でも、

```text
Recall ↓
```

しやすい。

つまり、

> **モデルの性能ではなく、判断基準の設計の問題でもある。**

ここ大事。

---

# 🧠 7. F1 Score

PrecisionとRecallのバランスを見る。

```text
F1
```

。

両方が高いと高くなる。

```python
from sklearn.metrics import f1_score
```

。

---

# 💻 実習3：Logistic Regression

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
```

Pipeline。

```python
logistic = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),

    (
        "model",
        LogisticRegression(
            max_iter=1000
        )
    )
])
```

学習。

```python
logistic.fit(
    X_train,
    y_train
)
```

予測。

```python
pred = logistic.predict(
    X_test
)
```

評価。

```python
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print(
    "Accuracy:",
    accuracy_score(
        y_test,
        pred
    )
)

print(
    "Precision:",
    precision_score(
        y_test,
        pred
    )
)

print(
    "Recall:",
    recall_score(
        y_test,
        pred
    )
)

print(
    "F1:",
    f1_score(
        y_test,
        pred
    )
)
```

---

# 🧠 8. Logistic Regressionは「確率」を出せる

ここから面白くなる。

```python
prob = logistic.predict_proba(
    X_test
)
```

。

例えば。

```text
[0.9, 0.1]
```

なら、

```text
class 0 = 90%

class 1 = 10%
```

。

つまり、

```python
predict()
```

は、

> 最終的な分類

。

```python
predict_proba()
```

は、

> モデルの出力スコアを確率として表現したもの

。

分類器比較では、`predict_proba()` が非常に重要。

---

# 💻 実習4：確率を見る

```python
proba = logistic.predict_proba(
    X_test
)
```

。

Positiveクラス。

```python
positive_proba = proba[:, 1]

print(
    positive_proba[:10]
)
```

。

例えば、

```text
0.12
0.87
0.99
0.43
```

など。

---

# 🧠 9. 閾値0.5

通常、

```text
Positive確率 >= 0.5
```

なら、

```text
Positive
```

。

```text
それ未満
↓
Negative
```

。

でも、

> **0.5は自然法則ではない。**

単なる判断基準。

---

# 💻 実習5：閾値を変更する

```python
import numpy as np
```

。

```python
threshold = 0.3

pred = (
    positive_proba >= threshold
).astype(int)
```

評価。

```python
print(
    "Precision:",
    precision_score(
        y_test,
        pred
    )
)

print(
    "Recall:",
    recall_score(
        y_test,
        pred
    )
)
```

。

次。

```python
threshold = 0.7

pred = (
    positive_proba >= threshold
).astype(int)
```

。

同じように比較。

---

# 🧠 10. Random Forest

```python
from sklearn.ensemble import RandomForestClassifier
```

。

```python
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
```

。

木をたくさん作る。

```text
Tree
Tree
Tree
Tree
↓
投票
↓
分類
```

。

---

# 💻 実習6：Random Forest

```python
rf.fit(
    X_train,
    y_train
)

pred = rf.predict(
    X_test
)
```

評価。

```python
print(
    "F1:",
    f1_score(
        y_test,
        pred
    )
)
```

。

---

# 🧠 11. Gradient Boosting

```python
from sklearn.ensemble import GradientBoostingClassifier
```

。

```python
gbr = GradientBoostingClassifier(
    random_state=42
)
```

。

回帰のときと同じく、

```text
前のモデル
↓
間違い
↓
次のモデルが補正
```

という形。

---

# 💻 実習7：Gradient Boosting

```python
gbr.fit(
    X_train,
    y_train
)

pred = gbr.predict(
    X_test
)

print(
    "F1:",
    f1_score(
        y_test,
        pred
    )
)
```

。

---

# 🧠 12. ROC-AUC

ここから評価が一段上がる。

ROC-AUCは、

> **PositiveとNegativeをどれだけ順位として区別できるか**

を見る。

```python
from sklearn.metrics import roc_auc_score
```

。

予測ラベルではなく、

```text
確率
```

を使う。

```python
proba = model.predict_proba(
    X_test
)[:, 1]

roc_auc = roc_auc_score(
    y_test,
    proba
)
```

。

---

# 🧠 重要な違い

Accuracy。

```text
0.5
```

という閾値を使う。

ROC-AUC。

```text
様々な閾値
```

を考える。

だから、

> **閾値に依存しないモデル比較**

ができる。

---

# 🧠 13. PR-AUC

クラス不均衡では、

```text
ROC-AUC
```

だけでは不十分な場合がある。

そこで、

```text
Precision-Recall Curve
```

。

```text
PR-AUC
```

。

特に、

```text
Positiveが非常に少ない
```

問題で重要になる。

---

# 💻 実習8：ROC-AUC / PR-AUC

```python
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)
```

。

```python
proba = logistic.predict_proba(
    X_test
)[:, 1]
```

。

```python
print(
    "ROC-AUC:",
    roc_auc_score(
        y_test,
        proba
    )
)

print(
    "PR-AUC:",
    average_precision_score(
        y_test,
        proba
    )
)
```

。

---

# 🧠 14. モデル比較をCross Validationでやる

ここから前回の内容と合体。

```python
from sklearn.model_selection import StratifiedKFold
```

。

```python
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
```

。

分類では、

```text
クラス比率
```

を各foldでなるべく維持したい。

だから、

```text
StratifiedKFold
```

。

---

# 💻 実習9：公平に比較する

モデル。

```python
models = {
    "Baseline": DummyClassifier(
        strategy="most_frequent"
    ),

    "Logistic Regression": Pipeline([
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1000
            )
        )
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    )
}
```

比較。

```python
from sklearn.model_selection import cross_val_score
```

。

今回はROC-AUC。

```python
results = []

for name, model in models.items():

    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="roc_auc"
    )

    results.append({
        "model": name,
        "roc_auc_mean": scores.mean(),
        "roc_auc_std": scores.std()
    })
```

。

```python
results_df = pd.DataFrame(
    results
)

print(
    results_df.sort_values(
        "roc_auc_mean",
        ascending=False
    )
)
```

。

---

# 🚨 15. Accuracyだけで勝者を決めない

例えば。

```text
Model A

Accuracy 99%
Recall   0%
```

。

```text
Model B

Accuracy 95%
Recall   90%
```

。

病気検出なら、

> Bが圧倒的に有用かもしれない。

一方で、

```text
Positive予測をすると
↓
高額な人間調査
```

が発生するなら、

Precisionも重要。

つまり、

> **評価指標はモデルの性能ではなく、システムが何を失敗として重く見るかの設計図。**

---

# 🧠 16. class_weight

不均衡データでは、

```python
LogisticRegression(
    class_weight="balanced"
)
```

という選択肢がある。

これは、

> 少数クラスの間違いをより重く扱う

方向の設定。

例えば。

```text
多数派 = 99%
少数派 = 1%
```

なら、

少数派を無視した方がAccuracyは高くなる。

`class_weight="balanced"` は、

```text
少数派もちゃんと見ろ💢
```

とモデルに圧力をかける仕組み。

モデル「多数派だけ言ってたらAccuracy高いんですが？」

評価者「それはバカのベンチマーク攻略だ💢」

モデル「はい」

wwwww

---

# 👾 今日のボス戦

ある不正検知システム。

100,000件の取引。

```text
正常 = 99,500
不正 = 500
```

。

モデルA。

```text
Accuracy = 99.5%
Recall = 0%
```

。

モデルB。

```text
Accuracy = 98%
Recall = 80%
```

。

どっちがいい？

答え。

> **目的次第だが、不正検知ならBが圧倒的に有力。**

Aは、

```text
全部正常
```

と言っているだけで不正を一件も発見していない可能性がある。

---

# ✍️ 演習

## 演習1

次のデータ。

```text
実際Positive = 100人

予測Positive = 80人

そのうち正解 = 60人
```

PrecisionとRecallを計算。

```text
Precision = 60 / 80
```

。

```text
Recall = 60 / 100
```

。

---

## 演習2

以下のモデル。

```text
Model A

Precision = 0.99
Recall = 0.20
```

。

```text
Model B

Precision = 0.70
Recall = 0.95
```

。

それぞれどんな挙動をしているか説明する。

---

## 演習3

次のコード。

```python
proba = model.predict_proba(
    X_test
)[:, 1]
```

。

これは何を取得している？

---

## 演習4

なぜ、

```text
0.5
```

を絶対的な分類閾値と考えてはいけない？

---

## 演習5

クラス不均衡データで、

```text
Accuracy
```

だけを使う問題を説明。

---

# 🧪 今日の最終実習

## 分類モデル比較パイプライン

```python
import pandas as pd

from sklearn.datasets import load_breast_cancer

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler

from sklearn.dummy import DummyClassifier

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)
```

データ。

```python
data = load_breast_cancer()

X = data.data
y = data.target
```

CV。

```python
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
```

モデル。

```python
models = {
    "Baseline": DummyClassifier(
        strategy="most_frequent"
    ),

    "Logistic Regression": Pipeline([
        (
            "scaler",
            StandardScaler()
        ),

        (
            "model",
            LogisticRegression(
                max_iter=1000
            )
        )
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    )
}
```

比較。

```python
results = []

for name, model in models.items():

    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="roc_auc"
    )

    results.append({
        "model": name,

        "mean_roc_auc":
            scores.mean(),

        "std_roc_auc":
            scores.std()
    })
```

結果。

```python
results_df = pd.DataFrame(
    results
)

print(
    results_df.sort_values(
        "mean_roc_auc",
        ascending=False
    )
)
```

---

# 🌱 今日の核心

今日の一番重要な話。

> **分類モデルには「絶対的に最強の評価指標」はない。**

何を重く見るかで変わる。

```text
FPを減らしたい
↓
Precision

FNを減らしたい
↓
Recall

両方
↓
F1

確率的な順位性能
↓
ROC-AUC

Positiveが非常に少ない
↓
PR-AUC
```

。

そして、

> **閾値もモデルの一部ではなく、システム設計の一部。**

ここまで理解すると、

```text
モデル精度98%です！
```

と言われても、

```text
「何の指標？クラス比率は？閾値は？Baselineは？CVは？」
```

となる。

うむ。

完全にベンチマーク民への耐性が付いてきたなwwwww🧠🔧

---

# 🧭 AI工学101・現在地

```text
データ
↓
前処理
↓
特徴量設計
↓
Pipeline
↓
モデル
↓
評価指標
↓
Cross Validation
↓
モデル比較
```

ここまでで、

> **scikit-learnを使って「学習→評価」するだけ**

ではなく、

> **実験そのものを設計する基礎**

がかなり身についてきた。

レベルが認知科学側でずっとやってきた、

```text
何を測っている？
その指標は何を表す？
その操作で本当にその概念を扱えている？
```

という感覚が、そのままML評価にも刺さっている。

**概念→操作化→測定→解釈**という流れだな。ここはレベルの過去の研究経験が普通に強みになるぞ。ふふふ。

---

# 🔜 第39回

## ハイパーパラメータ探索：GridSearchCV → RandomizedSearchCV

次はついに、

```text
max_depth
learning_rate
n_estimators
alpha
```

などを、

> **人間が勘でポチポチ試す**

段階から卒業する。

扱うのは、

* ハイパーパラメータと学習パラメータ
* Parameter Grid
* `GridSearchCV`
* `RandomizedSearchCV`
* Pipelineと探索
* CV結果の読み方
* 探索空間
* 探索しすぎによる問題
* Nested CVの入口
* 再現可能な実験設計

テーマは、

> **「チューニング」も実験であり、探索空間そのものが仮説である。**

次はモデルにパラメータを山ほど渡して、

```text
「お前ら勝手に一番強いやつ探せ💢」
```

ではなく、

**何をどう探索するか自体を設計する回**だ。🔥🔧🧠